In [1]:
import os
import pandas as pd
import numpy as np
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ==============================
# CONFIG — CHEMINS CORRECTS
# ==============================
W1 = r"C:\Users\INFOTEC\OneDrive\Bureau\Pre_w1w2\Cross_Week_Results\Week1_Final.xlsx"
W2 = r"C:\Users\INFOTEC\OneDrive\Bureau\Pre_w2w3\Cross_Week_Results\Week2_Final.xlsx"
W3 = r"C:\Users\INFOTEC\OneDrive\Bureau\pre_w3w4\Cross_Week_Results\Week3_Final.xlsx"
OUT_DIR = r"C:\Users\INFOTEC\OneDrive\Bureau\newton_week_to_days"

os.makedirs(OUT_DIR, exist_ok=True)

PAYS_LIST = ['Cyclam', 'Germany', 'India', 'Korea', 'Kunshan', 'Tianjin', 'USA', 'SAME', 'SCEET']

# ==============================
# COULEURS
# ==============================
C_HDR_BG     = '1F4E79'
C_HDR_FG     = 'FFFFFF'
C_SUM_BG     = '1A5276'
C_ROW_ALT    = ['EBF5FB', 'FDFEFE']
C_MISSING    = 'E8E8E8'   # Gris = produit absent dans cette semaine (qty=0)
C_PERTURB_OK = 'C6EFCE'   # Vert  = perturbation <= 2%
C_PERTURB_KO = 'FFC7CE'   # Rouge = perturbation >  2%
C_DAY_HDRS   = ['BDD7EE', 'FCE4D6', 'E2EFDA', 'FFF2CC', 'F4CCFF', 'D9EAD3']
WK_COLORS    = ['1A6BAD', '1A8A6B', '8A1A6B']
DAYS         = ['Lundi', 'Mardi', 'Mercredi', 'Jeudi', 'Vendredi', 'Samedi']

thin = Side(style='thin', color='BFBFBF')
BORD = Border(left=thin, right=thin, top=thin, bottom=thin)


# ==============================
# HELPERS
# ==============================
def sc(cell, bold=False, bg=None, fg='000000', align='center', wrap=False, fmt=None):
    cell.font      = Font(name='Arial', bold=bold, color=fg, size=9)
    if bg:
        cell.fill  = PatternFill('solid', start_color=bg)
    cell.alignment = Alignment(horizontal=align, vertical='center', wrap_text=wrap)
    cell.border    = BORD
    if fmt:
        cell.number_format = fmt


def to_f(v):
    try:
        return float(str(v).replace(',', '.').replace(' ', ''))
    except:
        return 0.0


# ==============================
# NEWTON — stock quotidien
# ==============================
def newton_daily_usage(inv1, wu1, inv2, wu2, inv3, wu3):
    """
    Calcule l'usage quotidien sur 6 jours par interpolation de Newton.
    Noeuds : (semaine1, inv1), (semaine2, inv2), (semaine3, inv3)
    Perturb% = (Usage_Jd - WU/6) / (WU/6) x 100
    """
    results = []
    for inv_start, wu in [(inv1, wu1), (inv2, wu2), (inv3, wu3)]:
        xs = np.linspace(1, 3, 6)
        d1_n  = (1.01 - 0.99)
        d2_n  = (0.99 - 1.01)
        d12_n = (d2_n - d1_n) / 2
        factors = 0.99 + d1_n * (xs - 1) + d12_n * (xs - 1) * (xs - 2)

        raw_usage = (wu / 6) * factors
        if raw_usage.sum() > 0:
            raw_usage = raw_usage * (wu / raw_usage.sum())

        stock_avant = []
        stock_apres = []
        s = inv_start
        for u in raw_usage:
            stock_avant.append(round(s, 4))
            s = max(0.0, s - u)
            stock_apres.append(round(s, 4))

        results.append({
            'stock_avant': stock_avant,
            'usage':       [round(a - b, 4) for a, b in zip(stock_avant, stock_apres)],
            'stock_apres': stock_apres,
            'wu':          wu,
            'inv_start':   inv_start,
        })
    return results


# ==============================
# CHARGEMENT — union des PNs
# ==============================
def load_pays(pays):
    """
    Charge les 3 semaines pour un pays.
    UNION complete des Part Numbers :
      - Ordre : PNs de W1 d'abord, puis nouveaux de W2, puis nouveaux de W3
      - PN absent d'une semaine => ligne ajoutee avec qty = 0 (fond gris)
    """
    NUM_COLS = ['Real Inventory (Qty)', 'Weekly Usage (Qty)',
                'Unit Price (€)', 'Stock Value (€)', 'Weekly Target']
    raws = []
    for wpath in [W1, W2, W3]:
        xl = pd.read_excel(wpath, sheet_name=pays, header=0)
        for c in NUM_COLS:
            if c in xl.columns:
                xl[c] = xl[c].apply(to_f)
        xl['Part Number'] = xl['Part Number'].astype(str).str.strip()
        raws.append(xl)

    # Union ordonnee des PNs
    seen = {}
    for df in raws:
        for _, row in df.iterrows():
            pn = str(row['Part Number']).strip()
            if pn not in seen:
                seen[pn] = {
                    'Description':    row.get('Description', ''),
                    'Supplier':       row.get('Supplier', ''),
                    'Unit Price (€)': to_f(row.get('Unit Price (€)', 0)),
                }
    all_pns_ordered = list(seen.keys())

    # Reconstruire chaque df avec tous les PNs (0 si absent)
    dfs = []
    for df in raws:
        rows = []
        for pn in all_pns_ordered:
            match = df[df['Part Number'].astype(str).str.strip() == pn]
            if not match.empty:
                r = match.iloc[0].to_dict()
                r['_missing'] = False
                rows.append(r)
            else:
                rows.append({
                    'Part Number':          pn,
                    'Description':          seen[pn]['Description'],
                    'Supplier':             seen[pn]['Supplier'],
                    'Unit Price (€)':       seen[pn]['Unit Price (€)'],
                    'Real Inventory (Qty)': 0.0,
                    'Weekly Usage (Qty)':   0.0,
                    'Stock Value (€)':      0.0,
                    'Weekly Target':        0.0,
                    '_missing':             True,
                })
        dfs.append(pd.DataFrame(rows).reset_index(drop=True))

    return dfs


# ==============================
# SHEET WEEK
# ==============================
def write_week_sheet(wb, pays, week_num, df_wk, all_newton):
    ws = wb.create_sheet(f'📅 Week{week_num}')
    ws.freeze_panes = 'E4'
    NCOLS = 38

    # Ligne 1 : Titre
    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=NCOLS)
    t = ws.cell(1, 1,
                f"📅  {pays} — Week {week_num}  |  Stock quotidien Newton  |  "
                f"Perturb% = (Usage_Jd - WU/6) / (WU/6)  |  "
                f"Gris = produit absent (qty=0)")
    sc(t, bold=True, bg=C_HDR_BG, fg=C_HDR_FG, align='left')
    ws.row_dimensions[1].height = 20

    # Ligne 2 : En-tetes groupes
    for c, lbl in zip([1, 2, 3, 4], ['Part Number', 'Description', 'Supplier', 'Unit Price (€)']):
        ws.merge_cells(start_row=2, start_column=c, end_row=3, end_column=c)
        sc(ws.cell(2, c, lbl), bold=True, bg=C_HDR_BG, fg=C_HDR_FG, wrap=True)

    ws.merge_cells(start_row=2, start_column=5, end_row=2, end_column=8)
    sc(ws.cell(2, 5, f'📌 Résumé Semaine {week_num}'), bold=True, bg='2E75B6', fg='FFFFFF')
    for c in [6, 7, 8]:
        sc(ws.cell(2, c), bg='2E75B6', fg='FFFFFF')

    for d in range(6):
        start_c = 9 + d * 5
        ws.merge_cells(start_row=2, start_column=start_c, end_row=2, end_column=start_c + 4)
        day_cell = ws.cell(2, start_c, f'J{d+1} — {DAYS[d]}')
        day_cell.font      = Font(name='Arial', bold=True, color='1F3864', size=9)
        day_cell.fill      = PatternFill('solid', start_color=C_DAY_HDRS[d])
        day_cell.alignment = Alignment(horizontal='center', vertical='center')
        day_cell.border    = BORD
        for c in range(start_c + 1, start_c + 5):
            cell = ws.cell(2, c)
            cell.fill   = PatternFill('solid', start_color=C_DAY_HDRS[d])
            cell.border = BORD
    ws.row_dimensions[2].height = 18

    # Ligne 3 : Sous-en-tetes
    for i, lbl in enumerate(['Inv. Réel', 'Usage Hebdo', 'Val. Stock', 'Cible']):
        cell = ws.cell(3, 5 + i, lbl)
        cell.font      = Font(name='Arial', bold=True, color='FFFFFF', size=9)
        cell.fill      = PatternFill('solid', start_color='1A5276')
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        cell.border    = BORD

    for d in range(6):
        for j, lbl in enumerate(['Avant', 'Usage', 'Après', 'Val.', 'Perturb%']):
            cell = ws.cell(3, 9 + d * 5 + j, lbl)
            cell.font      = Font(name='Arial', bold=True, color='1F3864', size=9)
            cell.fill      = PatternFill('solid', start_color=C_DAY_HDRS[d])
            cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
            cell.border    = BORD
    ws.row_dimensions[3].height = 28

    # Lignes de donnees
    for r_idx, row in df_wk.iterrows():
        pn         = str(row['Part Number']).strip()
        er         = r_idx + 4
        is_missing = row.get('_missing', False) == True
        bg         = C_MISSING if is_missing else C_ROW_ALT[r_idx % 2]
        up         = to_f(row.get('Unit Price (€)', 0))
        ri         = to_f(row.get('Real Inventory (Qty)', 0))
        wu         = to_f(row.get('Weekly Usage (Qty)', 0))
        sv         = to_f(row.get('Stock Value (€)', 0))
        wt         = to_f(row.get('Weekly Target', 0))

        def wc(col, val, fmt=None, bg_ov=None):
            cell = ws.cell(er, col, val)
            cell.font      = Font(name='Arial', size=9)
            cell.fill      = PatternFill('solid', start_color=bg_ov or bg)
            cell.alignment = Alignment(
                horizontal='right' if isinstance(val, (int, float)) else 'left',
                vertical='center')
            cell.border = BORD
            if fmt:
                cell.number_format = fmt

        wc(1, pn)
        wc(2, str(row.get('Description', '')) if pd.notna(row.get('Description', '')) else '')
        wc(3, str(row.get('Supplier', ''))    if pd.notna(row.get('Supplier', ''))    else '')
        wc(4, up, '#,##0.0000')
        wc(5, ri, '#,##0.00')
        wc(6, wu, '#,##0.00')
        wc(7, sv, '#,##0.00')
        wc(8, wt, '#,##0.00')

        nt  = all_newton.get(pn)
        wu6 = wu / 6 if wu != 0 else 0

        for d in range(6):
            base_c = 9 + d * 5
            if nt:
                wk_data = nt[week_num - 1]
                avant   = wk_data['stock_avant'][d]
                usage   = wk_data['usage'][d]
                apres   = wk_data['stock_apres'][d]
                sv_d    = round(apres * up, 4)
            else:
                avant = usage = apres = sv_d = 0.0

            perturb = ((usage - wu6) / wu6 * 100) if wu6 != 0 else 0.0

            wc(base_c,     avant, '#,##0.00')
            wc(base_c + 1, usage, '#,##0.00')
            wc(base_c + 2, apres, '#,##0.00')
            wc(base_c + 3, sv_d,  '#,##0.00')

            p_cell = ws.cell(er, base_c + 4, round(perturb, 2))
            p_cell.font      = Font(name='Arial', size=9)
            p_cell.fill      = PatternFill('solid',
                                           start_color=C_PERTURB_OK if abs(perturb) <= 2 else C_PERTURB_KO)
            p_cell.alignment = Alignment(horizontal='right', vertical='center')
            p_cell.border    = BORD
            p_cell.number_format = '0.00"%"'

    ws.column_dimensions['A'].width = 14
    ws.column_dimensions['B'].width = 28
    ws.column_dimensions['C'].width = 22
    ws.column_dimensions['D'].width = 11
    for c in range(5, NCOLS + 1):
        ws.column_dimensions[get_column_letter(c)].width = 11


# ==============================
# SHEET SUMMARY
# ==============================
def write_summary(wb, pays, dfs, all_newton):
    ws = wb.create_sheet('📋 Summary', 0)
    ws.freeze_panes = 'E4'
    NCOLS = 16

    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=NCOLS)
    t = ws.cell(1, 1,
                f'{pays} — Summary 3 Semaines  |  '
                f'Inv. Réel · Usage · Val. Stock · Perturbation% Moyenne  |  '
                f'Gris = produit absent (qty=0)')
    t.font      = Font(name='Arial', bold=True, color='FFFFFF', size=10)
    t.fill      = PatternFill('solid', start_color=C_HDR_BG)
    t.alignment = Alignment(horizontal='left', vertical='center')
    t.border    = BORD
    ws.row_dimensions[1].height = 20

    for c, lbl in zip([1, 2, 3, 4], ['Part Number', 'Description', 'Supplier', 'Unit Price (€)']):
        ws.merge_cells(start_row=2, start_column=c, end_row=3, end_column=c)
        cell = ws.cell(2, c, lbl)
        cell.font      = Font(name='Arial', bold=True, color='FFFFFF', size=9)
        cell.fill      = PatternFill('solid', start_color=C_HDR_BG)
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        cell.border    = BORD

    for w in range(3):
        start_c = 5 + w * 4
        ws.merge_cells(start_row=2, start_column=start_c, end_row=2, end_column=start_c + 3)
        cell = ws.cell(2, start_c, f'Week {w+1}')
        cell.font      = Font(name='Arial', bold=True, color='FFFFFF', size=9)
        cell.fill      = PatternFill('solid', start_color=WK_COLORS[w])
        cell.alignment = Alignment(horizontal='center', vertical='center')
        cell.border    = BORD
        for c in range(start_c + 1, start_c + 4):
            cell = ws.cell(2, c)
            cell.fill   = PatternFill('solid', start_color=WK_COLORS[w])
            cell.border = BORD
    ws.row_dimensions[2].height = 18

    for w in range(3):
        for j, lbl in enumerate(['Inv. Réel', 'Usage', 'Val. Stock', 'Perturb% moy.']):
            cell = ws.cell(3, 5 + w * 4 + j, lbl)
            cell.font      = Font(name='Arial', bold=True, color='FFFFFF', size=9)
            cell.fill      = PatternFill('solid', start_color=WK_COLORS[w])
            cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
            cell.border    = BORD
    ws.row_dimensions[3].height = 30

    all_pns = list(dfs[0]['Part Number'].astype(str).str.strip())

    for r_idx, pn in enumerate(all_pns):
        er   = r_idx + 4
        row1 = dfs[0][dfs[0]['Part Number'].astype(str).str.strip() == pn]

        is_missing_all = all(
            dfs[w][dfs[w]['Part Number'].astype(str).str.strip() == pn].iloc[0].get('_missing', False) == True
            if not dfs[w][dfs[w]['Part Number'].astype(str).str.strip() == pn].empty else True
            for w in range(3)
        )
        bg = C_MISSING if is_missing_all else C_ROW_ALT[r_idx % 2]

        def wc(col, val, fmt=None, bg_ov=None):
            cell = ws.cell(er, col, val)
            cell.font      = Font(name='Arial', size=9)
            cell.fill      = PatternFill('solid', start_color=bg_ov or bg)
            cell.alignment = Alignment(
                horizontal='right' if isinstance(val, (int, float)) else 'left',
                vertical='center')
            cell.border = BORD
            if fmt:
                cell.number_format = fmt

        if not row1.empty:
            r = row1.iloc[0]
            wc(1, pn)
            wc(2, str(r.get('Description', '')) if pd.notna(r.get('Description', '')) else '')
            wc(3, str(r.get('Supplier', ''))    if pd.notna(r.get('Supplier', ''))    else '')
            wc(4, to_f(r.get('Unit Price (€)', 0)), '#,##0.0000')

        nt = all_newton.get(pn)
        for w in range(3):
            base_c = 5 + w * 4
            row_w  = dfs[w][dfs[w]['Part Number'].astype(str).str.strip() == pn]
            if not row_w.empty:
                r   = row_w.iloc[0]
                ri  = to_f(r.get('Real Inventory (Qty)', 0))
                wu  = to_f(r.get('Weekly Usage (Qty)', 0))
                sv  = to_f(r.get('Stock Value (€)', 0))
                wu6 = wu / 6 if wu != 0 else 0
                if nt and wu6 != 0:
                    usages   = nt[w]['usage']
                    perturbs = [(u - wu6) / wu6 * 100 for u in usages]
                    pb = round(np.mean(perturbs), 2)
                else:
                    pb = 0.0
                wc(base_c,     ri, '#,##0.00')
                wc(base_c + 1, wu, '#,##0.00')
                wc(base_c + 2, sv, '#,##0.00')
                p_cell = ws.cell(er, base_c + 3, pb)
                p_cell.font      = Font(name='Arial', size=9)
                p_cell.fill      = PatternFill('solid',
                                               start_color=C_PERTURB_OK if abs(pb) <= 2 else C_PERTURB_KO)
                p_cell.alignment = Alignment(horizontal='right', vertical='center')
                p_cell.border    = BORD
                p_cell.number_format = '0.00"%"'
            else:
                for c in range(base_c, base_c + 4):
                    wc(c, '-')

    # Ligne TOTAL / MOYENNE
    tot_row = len(all_pns) + 4
    ws.merge_cells(start_row=tot_row, start_column=1, end_row=tot_row, end_column=4)
    cell = ws.cell(tot_row, 1, 'TOTAL / MOYENNE')
    cell.font      = Font(name='Arial', bold=True, color='FFFFFF', size=9)
    cell.fill      = PatternFill('solid', start_color=C_SUM_BG)
    cell.alignment = Alignment(horizontal='center', vertical='center')
    cell.border    = BORD

    for w in range(3):
        for j in range(4):
            c     = 5 + w * 4 + j
            col_l = get_column_letter(c)
            formula = (f'=AVERAGE({col_l}4:{col_l}{tot_row-1})'
                       if j == 3 else
                       f'=SUM({col_l}4:{col_l}{tot_row-1})')
            fmt   = '0.00"%"' if j == 3 else '#,##0.00'
            cell  = ws.cell(tot_row, c, formula)
            cell.font      = Font(name='Arial', bold=True, color='FFFFFF', size=9)
            cell.fill      = PatternFill('solid', start_color=C_SUM_BG)
            cell.alignment = Alignment(horizontal='right', vertical='center')
            cell.border    = BORD
            cell.number_format = fmt

    ws.column_dimensions['A'].width = 14
    ws.column_dimensions['B'].width = 28
    ws.column_dimensions['C'].width = 22
    ws.column_dimensions['D'].width = 11
    for c in range(5, NCOLS + 1):
        ws.column_dimensions[get_column_letter(c)].width = 13


# ==============================
# MAIN
# ==============================
def main():
    for pays in PAYS_LIST:
        print(f'⏳ Processing {pays}...')
        try:
            dfs = load_pays(pays)
        except Exception as e:
            print(f'  ⚠️  Skipped {pays}: {e}')
            continue

        df1, df2, df3 = dfs

        all_newton = {}
        all_pns    = list(df1['Part Number'].astype(str).str.strip())

        for pn in all_pns:
            def get_val(df, pn_=pn, col=None):
                r = df[df['Part Number'].astype(str).str.strip() == pn_]
                return to_f(r.iloc[0][col]) if not r.empty else 0.0

            inv1 = get_val(df1, col='Real Inventory (Qty)')
            wu1  = get_val(df1, col='Weekly Usage (Qty)')
            inv2 = get_val(df2, col='Real Inventory (Qty)')
            wu2  = get_val(df2, col='Weekly Usage (Qty)')
            inv3 = get_val(df3, col='Real Inventory (Qty)')
            wu3  = get_val(df3, col='Weekly Usage (Qty)')

            all_newton[pn] = newton_daily_usage(inv1, wu1, inv2, wu2, inv3, wu3)

        wb = Workbook()
        wb.remove(wb.active)

        write_summary(wb, pays, dfs, all_newton)
        write_week_sheet(wb, pays, 1, df1, all_newton)
        write_week_sheet(wb, pays, 2, df2, all_newton)
        write_week_sheet(wb, pays, 3, df3, all_newton)

        out_path = os.path.join(OUT_DIR, f'{pays}_Stock_Quotidien_Full.xlsx')
        wb.save(out_path)
        print(f'  ✅ Saved: {out_path}')

    print('\n🎯 TERMINÉ — Tous les pays générés!')


main()

⏳ Processing Cyclam...
  ✅ Saved: C:\Users\INFOTEC\OneDrive\Bureau\newton_week_to_days\Cyclam_Stock_Quotidien_Full.xlsx
⏳ Processing Germany...
  ✅ Saved: C:\Users\INFOTEC\OneDrive\Bureau\newton_week_to_days\Germany_Stock_Quotidien_Full.xlsx
⏳ Processing India...
  ✅ Saved: C:\Users\INFOTEC\OneDrive\Bureau\newton_week_to_days\India_Stock_Quotidien_Full.xlsx
⏳ Processing Korea...
  ✅ Saved: C:\Users\INFOTEC\OneDrive\Bureau\newton_week_to_days\Korea_Stock_Quotidien_Full.xlsx
⏳ Processing Kunshan...
  ✅ Saved: C:\Users\INFOTEC\OneDrive\Bureau\newton_week_to_days\Kunshan_Stock_Quotidien_Full.xlsx
⏳ Processing Tianjin...
  ✅ Saved: C:\Users\INFOTEC\OneDrive\Bureau\newton_week_to_days\Tianjin_Stock_Quotidien_Full.xlsx
⏳ Processing USA...
  ✅ Saved: C:\Users\INFOTEC\OneDrive\Bureau\newton_week_to_days\USA_Stock_Quotidien_Full.xlsx
⏳ Processing SAME...
  ✅ Saved: C:\Users\INFOTEC\OneDrive\Bureau\newton_week_to_days\SAME_Stock_Quotidien_Full.xlsx
⏳ Processing SCEET...
  ✅ Saved: C:\Users\INFOTE

In [2]:
import os
import pandas as pd
import numpy as np
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ==============================
# CONFIG
# ==============================
W1 = r"C:\Users\INFOTEC\OneDrive\Bureau\anomalieW1\anomalie_metier\Week1_With_Anomalies_Metier.xlsx"
W2 = r"C:\Users\INFOTEC\OneDrive\Bureau\anomalie_w2\anomalie_metier\Week2_With_Anomalies_Metier.xlsx"
W3 = r"C:\Users\INFOTEC\OneDrive\Bureau\anomalie_w3\anomalie_metier\Week3_With_Anomalies_Metier.xlsx"
OUT_DIR = r"C:\Users\INFOTEC\OneDrive\Bureau\newton_week_to_day"
os.makedirs(OUT_DIR, exist_ok=True)

PAYS_LIST = ['Cyclam', 'Germany', 'India', 'Korea', 'Kunshan', 'Tianjin', 'USA', 'SAME', 'SCEET']
DAYS      = ['Lundi', 'Mardi', 'Mercredi', 'Jeudi', 'Vendredi', 'Samedi']

# ==============================
# COULEURS
# ==============================
C_HDR_BG   = '1F4E79'
C_HDR_FG   = 'FFFFFF'
C_SUM_BG   = '1A5276'
C_ROW_ALT  = ['EBF5FB', 'FDFEFE']
C_MISSING  = 'E8E8E8'
C_PERT_OK  = 'C6EFCE'
C_PERT_KO  = 'FFC7CE'
C_DAY_HDRS = ['BDD7EE', 'FCE4D6', 'E2EFDA', 'FFF2CC', 'F4CCFF', 'D9EAD3']
WK_COLORS  = ['1A6BAD', '1A8A6B', '8A1A6B']

# Anomalie metier colors
ANOM_COLORS = {
    'Normal':             ('F2F2F2', '000000'),
    'Rupture de stock':   ('FFC7CE', 'C00000'),
    'Sur-stock critique': ('FFEB9C', '9C6500'),
    'Rotation trop rapide': ('BDD7EE', '17375E'),
    'Sous-stock':         ('FCE4D6', '833C00'),
}

thin = Side(style='thin', color='BFBFBF')
BORD = Border(left=thin, right=thin, top=thin, bottom=thin)


def to_f(v):
    try:
        return float(str(v).replace(',', '.').replace(' ', '').replace('\xa0', ''))
    except:
        return 0.0


def sc(cell, bold=False, bg=None, fg='000000', align='center', wrap=False, fmt=None, size=9):
    cell.font      = Font(name='Arial', bold=bold, color=fg, size=size)
    if bg:
        cell.fill  = PatternFill('solid', start_color=bg)
    cell.alignment = Alignment(horizontal=align, vertical='center', wrap_text=wrap)
    cell.border    = BORD
    if fmt:
        cell.number_format = fmt


def newton_daily(inv_start, wu):
    xs    = np.linspace(1, 3, 6)
    d1    = 1.01 - 0.99
    d2    = 0.99 - 1.01
    d12   = (d2 - d1) / 2
    facs  = 0.99 + d1 * (xs - 1) + d12 * (xs - 1) * (xs - 2)
    raw   = (wu / 6) * facs
    if raw.sum() > 0:
        raw = raw * (wu / raw.sum())
    avant, apres = [], []
    s = inv_start
    for u in raw:
        avant.append(round(s, 4))
        s = max(0.0, s - u)
        apres.append(round(s, 4))
    usage = [round(a - b, 4) for a, b in zip(avant, apres)]
    return avant, usage, apres


def get_anom_cols(df_cols):
    """Return normalized anomaly detail column mapping"""
    mapping = {}
    for c in df_cols:
        cl = c.strip()
        if 'Inventory_Negatif' in cl or 'Inventory_Neg' in cl:
            mapping['inv_neg'] = cl
        elif 'Rupture_Stock' in cl or 'Rupture' in cl:
            mapping['rupture'] = cl
        elif 'Rotation_Rapide' in cl or 'Rotation' in cl:
            mapping['rotation'] = cl
        elif 'Sous_Stock' in cl:
            mapping['sous_stock'] = cl
        elif 'Surstock_Critique' in cl or 'Surstock' in cl:
            mapping['surstock'] = cl
        elif 'Total_Anomalies' in cl:
            mapping['total'] = cl
        elif cl == 'Anomalie_Metier':
            mapping['metier'] = cl
    return mapping


def load_pays_data(pays):
    """Load W1, W2, W3 for a pays — return (df1, df2, df3) with union of PNs"""
    raws = []
    for wpath in [W1, W2, W3]:
        xl = pd.read_excel(wpath, sheet_name=pays, header=0)
        xl.columns = xl.columns.str.strip()
        xl['Part Number'] = xl['Part Number'].astype(str).str.strip()
        raws.append(xl)

    # Union ordered PNs
    seen = {}
    for df in raws:
        for _, row in df.iterrows():
            pn = str(row['Part Number']).strip()
            if pn not in seen:
                seen[pn] = {
                    'Description':    row.get('Description', ''),
                    'Supplier':       row.get('Supplier', ''),
                    'Unit Price (€)': to_f(row.get('Unit Price (€)', 0)),
                }
    all_pns = list(seen.keys())

    dfs = []
    for df in raws:
        rows = []
        for pn in all_pns:
            match = df[df['Part Number'] == pn]
            if not match.empty:
                r = match.iloc[0].to_dict()
                r['_missing'] = False
            else:
                r = {
                    'Part Number': pn,
                    'Description': seen[pn]['Description'],
                    'Supplier':    seen[pn]['Supplier'],
                    'Unit Price (€)': seen[pn]['Unit Price (€)'],
                    'Real Inventory (Qty)': 0.0,
                    'Weekly Usage (Qty)':   0.0,
                    'Stock Value (€)':      0.0,
                    'Weekly Target':        0.0,
                    '_missing': True,
                }
            rows.append(r)
        dfs.append(pd.DataFrame(rows).reset_index(drop=True))
    return dfs, all_pns, seen


# ==============================
# WRITE WEEK SHEET
# ==============================
def write_week_sheet(wb, pays, week_num, df_wk, all_newton):
    ws = wb.create_sheet(f'Week{week_num}')
    ws.freeze_panes = 'E4'

    # Anomaly columns present in this week's df
    amap = get_anom_cols(df_wk.columns.tolist())
    detail_keys = ['inv_neg', 'rupture', 'rotation', 'sous_stock', 'surstock', 'total']
    detail_labels = ['Inv. Neg', 'Rupture', 'Rotation rapide', 'Sous-stock', 'Sur-stock', 'Total Anom.']
    present_details = [(k, detail_labels[i]) for i, k in enumerate(detail_keys) if k in amap]

    # Column layout:
    # 1=PN, 2=Desc, 3=Supplier, 4=UP
    # 5=Inv Réel, 6=WU, 7=Val Stock, 8=Target
    # 9=Anomalie Metier, 10..=detail anom cols
    # then days start
    N_ANOM_DETAIL = len(present_details)
    DAY_START = 10 + N_ANOM_DETAIL  # col where day data starts
    NCOLS = DAY_START - 1 + 6 * 5   # 5 sub-cols per day

    # Row 1: Title
    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=NCOLS)
    t = ws.cell(1, 1, f'{pays} — Week {week_num} | Stock Quotidien Newton | Anomalies Metier')
    sc(t, bold=True, bg=C_HDR_BG, fg=C_HDR_FG, align='left', size=10)
    ws.row_dimensions[1].height = 20

    # Row 2: Group headers
    for c, lbl in [(1,'Part Number'),(2,'Description'),(3,'Supplier'),(4,'Unit Price (€)')]:
        ws.merge_cells(start_row=2, start_column=c, end_row=3, end_column=c)
        sc(ws.cell(2, c, lbl), bold=True, bg=C_HDR_BG, fg=C_HDR_FG, wrap=True)
        sc(ws.cell(3, c), bg=C_HDR_BG)

    # Summary week block
    ws.merge_cells(start_row=2, start_column=5, end_row=2, end_column=8)
    sc(ws.cell(2, 5, f'Resume S{week_num}'), bold=True, bg='2E75B6', fg='FFFFFF')
    for c in [6,7,8]:
        sc(ws.cell(2, c), bg='2E75B6')

    # Anomalie Metier col
    ws.merge_cells(start_row=2, start_column=9, end_row=2, end_column=9 + N_ANOM_DETAIL)
    sc(ws.cell(2, 9, 'Anomalies Metier'), bold=True, bg='C00000', fg='FFFFFF')
    for c in range(10, 10 + N_ANOM_DETAIL):
        sc(ws.cell(2, c), bg='C00000')

    # Day headers
    for d in range(6):
        sc_ = DAY_START + d * 5
        ws.merge_cells(start_row=2, start_column=sc_, end_row=2, end_column=sc_ + 4)
        day_cell = ws.cell(2, sc_, f'J{d+1} — {DAYS[d]}')
        day_cell.font      = Font(name='Arial', bold=True, color='1F3864', size=9)
        day_cell.fill      = PatternFill('solid', start_color=C_DAY_HDRS[d])
        day_cell.alignment = Alignment(horizontal='center', vertical='center')
        day_cell.border    = BORD
        for c in range(sc_+1, sc_+5):
            ws.cell(2, c).fill   = PatternFill('solid', start_color=C_DAY_HDRS[d])
            ws.cell(2, c).border = BORD
    ws.row_dimensions[2].height = 18

    # Row 3: Sub-headers
    for i, lbl in enumerate(['Inv. Reel','Usage Hebdo','Val. Stock','Cible']):
        sc(ws.cell(3, 5+i, lbl), bold=True, bg='1A5276', fg='FFFFFF', wrap=True)

    sc(ws.cell(3, 9, 'Type'), bold=True, bg='C00000', fg='FFFFFF', wrap=True)
    for i, (_, lbl) in enumerate(present_details):
        sc(ws.cell(3, 10+i, lbl), bold=True, bg='C00000', fg='FFFFFF', wrap=True, size=8)

    for d in range(6):
        for j, lbl in enumerate(['Avant','Usage','Apres','Valeur','Perturb%']):
            c = DAY_START + d*5 + j
            cell = ws.cell(3, c, lbl)
            cell.font      = Font(name='Arial', bold=True, color='1F3864', size=9)
            cell.fill      = PatternFill('solid', start_color=C_DAY_HDRS[d])
            cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
            cell.border    = BORD
    ws.row_dimensions[3].height = 28

    # Data rows
    for r_idx, row in df_wk.iterrows():
        pn         = str(row.get('Part Number', '')).strip()
        er         = r_idx + 4
        is_missing = row.get('_missing', False) == True
        bg         = C_MISSING if is_missing else C_ROW_ALT[r_idx % 2]
        up         = to_f(row.get('Unit Price (€)', 0))
        ri         = to_f(row.get('Real Inventory (Qty)', 0))
        wu         = to_f(row.get('Weekly Usage (Qty)', 0))
        sv         = to_f(row.get('Stock Value (€)', 0))
        wt         = to_f(row.get('Weekly Target', 0))

        def wc(col, val, fmt=None, bg_ov=None, fg_ov=None, bold=False):
            cell = ws.cell(er, col, val)
            cell.font      = Font(name='Arial', size=9, color=fg_ov or '000000', bold=bold)
            cell.fill      = PatternFill('solid', start_color=bg_ov or bg)
            cell.alignment = Alignment(
                horizontal='right' if isinstance(val, (int, float)) else 'left',
                vertical='center')
            cell.border = BORD
            if fmt:
                cell.number_format = fmt

        wc(1, pn)
        wc(2, str(row.get('Description', '')) if pd.notna(row.get('Description', '')) else '')
        wc(3, str(row.get('Supplier', ''))    if pd.notna(row.get('Supplier', ''))    else '')
        wc(4, up, '#,##0.0000')
        wc(5, ri, '#,##0.00')
        wc(6, wu, '#,##0.00')
        wc(7, sv, '#,##0.00')
        wc(8, wt, '#,##0.00')

        # Anomalie Metier
        metier_val = ''
        if not is_missing and 'metier' in amap:
            v = row.get(amap['metier'], '')
            metier_val = str(v) if pd.notna(v) else ''
        anom_bg, anom_fg = ANOM_COLORS.get(metier_val, (bg, '000000'))
        if is_missing:
            anom_bg = C_MISSING
        wc(9, metier_val, bg_ov=anom_bg, fg_ov=anom_fg, bold=(metier_val not in ('', 'Normal')))

        # Detail flags
        for i, (key, _) in enumerate(present_details):
            col_name = amap[key]
            v = 0
            if not is_missing and col_name in row.index:
                v = to_f(row.get(col_name, 0))
            flag_bg = anom_bg if v == 1 else (C_MISSING if is_missing else bg)
            wc(10+i, int(v) if v in (0,1) else v, bg_ov=flag_bg)

        # Newton days
        nt  = all_newton.get(pn)
        wu6 = wu / 6 if wu != 0 else 0
        for d in range(6):
            bc = DAY_START + d * 5
            if nt:
                av, us, ap = nt
                avant = av[d]
                usage = us[d]
                apres = ap[d]
                sv_d  = round(apres * up, 4)
            else:
                avant = usage = apres = sv_d = 0.0
            perturb = ((usage - wu6) / wu6 * 100) if wu6 != 0 else 0.0

            wc(bc,   avant, '#,##0.00')
            wc(bc+1, usage, '#,##0.00')
            wc(bc+2, apres, '#,##0.00')
            wc(bc+3, sv_d,  '#,##0.00')
            pc = ws.cell(er, bc+4, round(perturb, 2))
            pc.font      = Font(name='Arial', size=9)
            pc.fill      = PatternFill('solid', start_color=C_PERT_OK if abs(perturb) <= 2 else C_PERT_KO)
            pc.alignment = Alignment(horizontal='right', vertical='center')
            pc.border    = BORD
            pc.number_format = '0.00'

    # Column widths
    ws.column_dimensions['A'].width = 16
    ws.column_dimensions['B'].width = 30
    ws.column_dimensions['C'].width = 22
    ws.column_dimensions['D'].width = 11
    ws.column_dimensions['E'].width = 11
    ws.column_dimensions['F'].width = 11
    ws.column_dimensions['G'].width = 11
    ws.column_dimensions['H'].width = 10
    ws.column_dimensions['I'].width = 20  # Anomalie Metier
    for i in range(N_ANOM_DETAIL):
        ws.column_dimensions[get_column_letter(10+i)].width = 11
    for c in range(DAY_START, NCOLS+1):
        ws.column_dimensions[get_column_letter(c)].width = 11


# ==============================
# WRITE SUMMARY SHEET
# ==============================
def write_summary(wb, pays, dfs, all_pns, newton_all):
    ws = wb.create_sheet('Summary', 0)
    ws.freeze_panes = 'E4'

    # Build anomaly maps per week (pn -> metier type)
    anom_maps = []
    for w, df in enumerate(dfs):
        df_c = df.copy()
        df_c.columns = [str(c).strip() for c in df_c.columns]
        amap = get_anom_cols(df_c.columns.tolist())
        m = {}
        if 'metier' in amap:
            for _, row in df_c.iterrows():
                pn = str(row.get('Part Number', '')).strip()
                v  = row.get(amap['metier'], '')
                m[pn] = str(v) if pd.notna(v) else ''
        anom_maps.append(m)

    # NCOLS: 4 info + 3*(4 stock + 1 anom) = 4 + 15 = 19
    NCOLS = 19
    WK_LBLS = ['W1', 'W2', 'W3']

    # Row 1: title
    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=NCOLS)
    t = ws.cell(1, 1, f'{pays} — Summary 3 Semaines | Stock + Anomalies Metier')
    sc(t, bold=True, bg=C_HDR_BG, fg=C_HDR_FG, align='left', size=10)
    ws.row_dimensions[1].height = 20

    for c, lbl in [(1,'Part Number'),(2,'Description'),(3,'Supplier'),(4,'Unit Price')]:
        ws.merge_cells(start_row=2, start_column=c, end_row=3, end_column=c)
        sc(ws.cell(2, c, lbl), bold=True, bg=C_HDR_BG, fg=C_HDR_FG, wrap=True)
        sc(ws.cell(3, c), bg=C_HDR_BG)

    for w in range(3):
        sc_ = 5 + w * 5
        ws.merge_cells(start_row=2, start_column=sc_, end_row=2, end_column=sc_+4)
        sc(ws.cell(2, sc_, f'Week {w+1}'), bold=True, bg=WK_COLORS[w], fg='FFFFFF')
        for c in range(sc_+1, sc_+5):
            sc(ws.cell(2, c), bg=WK_COLORS[w])
    ws.row_dimensions[2].height = 18

    for w in range(3):
        sc_ = 5 + w * 5
        for j, lbl in enumerate(['Inv. Reel','Usage','Val. Stock','Perturb% moy','Anomalie']):
            sc(ws.cell(3, sc_+j, lbl), bold=True, bg=WK_COLORS[w], fg='FFFFFF', wrap=True)
    ws.row_dimensions[3].height = 30

    for r_idx, pn in enumerate(all_pns):
        er  = r_idx + 4
        # check if missing all weeks
        miss_all = all(
            dfs[w][dfs[w]['Part Number'].astype(str).str.strip() == pn].iloc[0].get('_missing', False) == True
            if not dfs[w][dfs[w]['Part Number'].astype(str).str.strip() == pn].empty else True
            for w in range(3)
        )
        bg = C_MISSING if miss_all else C_ROW_ALT[r_idx % 2]

        def wc(col, val, fmt=None, bg_ov=None, fg_ov=None, bold=False):
            cell = ws.cell(er, col, val)
            cell.font      = Font(name='Arial', size=9, color=fg_ov or '000000', bold=bold)
            cell.fill      = PatternFill('solid', start_color=bg_ov or bg)
            cell.alignment = Alignment(
                horizontal='right' if isinstance(val, (int, float)) else 'left',
                vertical='center')
            cell.border = BORD
            if fmt:
                cell.number_format = fmt

        row1 = dfs[0][dfs[0]['Part Number'].astype(str).str.strip() == pn]
        if not row1.empty:
            r = row1.iloc[0]
            wc(1, pn)
            wc(2, str(r.get('Description','')) if pd.notna(r.get('Description','')) else '')
            wc(3, str(r.get('Supplier',''))    if pd.notna(r.get('Supplier',''))    else '')
            wc(4, to_f(r.get('Unit Price (€)',0)), '#,##0.0000')

        for w in range(3):
            sc_ = 5 + w * 5
            row_w = dfs[w][dfs[w]['Part Number'].astype(str).str.strip() == pn]
            if not row_w.empty:
                rw  = row_w.iloc[0]
                ri  = to_f(rw.get('Real Inventory (Qty)', 0))
                wu  = to_f(rw.get('Weekly Usage (Qty)',   0))
                sv  = to_f(rw.get('Stock Value (€)',      0))
                wu6 = wu / 6 if wu != 0 else 0
                nt  = newton_all[w].get(pn)
                if nt and wu6 != 0:
                    perturbs = [((u - wu6) / wu6 * 100) for u in nt[1]]
                    pb = round(np.mean(perturbs), 2)
                else:
                    pb = 0.0
                wc(sc_,   ri, '#,##0.00')
                wc(sc_+1, wu, '#,##0.00')
                wc(sc_+2, sv, '#,##0.00')
                # Perturb
                pc = ws.cell(er, sc_+3, pb)
                pc.font      = Font(name='Arial', size=9)
                pc.fill      = PatternFill('solid', start_color=C_PERT_OK if abs(pb) <= 2 else C_PERT_KO)
                pc.alignment = Alignment(horizontal='right', vertical='center')
                pc.border    = BORD
                pc.number_format = '0.00'
                # Anomalie metier
                anom_val = anom_maps[w].get(pn, '')
                ab, af = ANOM_COLORS.get(anom_val, (bg, '000000'))
                if rw.get('_missing', False):
                    ab = C_MISSING
                wc(sc_+4, anom_val, bg_ov=ab, fg_ov=af, bold=(anom_val not in ('','Normal')))
            else:
                for c in range(sc_, sc_+5):
                    wc(c, '-')

    # Total row
    tot_row = len(all_pns) + 4
    ws.merge_cells(start_row=tot_row, start_column=1, end_row=tot_row, end_column=4)
    sc(ws.cell(tot_row, 1, 'TOTAL / MOYENNE'), bold=True, bg=C_SUM_BG, fg='FFFFFF', align='center')
    for w in range(3):
        sc_ = 5 + w * 5
        for j in range(4):
            c     = sc_ + j
            col_l = get_column_letter(c)
            formula = (f'=AVERAGE({col_l}4:{col_l}{tot_row-1})' if j == 3
                       else f'=SUM({col_l}4:{col_l}{tot_row-1})')
            cell = ws.cell(tot_row, c, formula)
            cell.font      = Font(name='Arial', bold=True, color='FFFFFF', size=9)
            cell.fill      = PatternFill('solid', start_color=C_SUM_BG)
            cell.alignment = Alignment(horizontal='right', vertical='center')
            cell.border    = BORD
            cell.number_format = '0.00'
        wc_ = ws.cell(tot_row, sc_+4, '-')
        wc_.font  = Font(name='Arial', bold=True, color='FFFFFF', size=9)
        wc_.fill  = PatternFill('solid', start_color=C_SUM_BG)
        wc_.border = BORD
        wc_.alignment = Alignment(horizontal='center', vertical='center')

    ws.column_dimensions['A'].width = 16
    ws.column_dimensions['B'].width = 30
    ws.column_dimensions['C'].width = 22
    ws.column_dimensions['D'].width = 11
    for c in range(5, NCOLS+1):
        ws.column_dimensions[get_column_letter(c)].width = 14


# ==============================
# MAIN
# ==============================
def main():
    for pays in PAYS_LIST:
        print(f'Processing {pays}...')
        try:
            dfs, all_pns, seen = load_pays_data(pays)
        except Exception as e:
            print(f'  SKIP {pays}: {e}')
            continue

        df1, df2, df3 = dfs

        # Build newton per week separately (each week has own inv+wu)
        newton_all = [{}, {}, {}]
        for pn in all_pns:
            for w, df in enumerate(dfs):
                row = df[df['Part Number'].astype(str).str.strip() == pn]
                if not row.empty:
                    ri = to_f(row.iloc[0].get('Real Inventory (Qty)', 0))
                    wu = to_f(row.iloc[0].get('Weekly Usage (Qty)',   0))
                else:
                    ri = wu = 0.0
                avant, usage, apres = newton_daily(ri, wu)
                newton_all[w][pn] = (avant, usage, apres)

        wb = Workbook()
        wb.remove(wb.active)

        write_summary(wb, pays, dfs, all_pns, newton_all)
        write_week_sheet(wb, pays, 1, df1, newton_all[0])
        write_week_sheet(wb, pays, 2, df2, newton_all[1])
        write_week_sheet(wb, pays, 3, df3, newton_all[2])

        out_path = os.path.join(OUT_DIR, f'{pays}_Stock_Quotidien.xlsx')
        wb.save(out_path)
        print(f'  Saved: {out_path}  ({len(all_pns)} produits)')

    print('\nDone!')


main()

Processing Cyclam...
  Saved: C:\Users\INFOTEC\OneDrive\Bureau\newton_week_to_day\Cyclam_Stock_Quotidien.xlsx  (952 produits)
Processing Germany...
  Saved: C:\Users\INFOTEC\OneDrive\Bureau\newton_week_to_day\Germany_Stock_Quotidien.xlsx  (167 produits)
Processing India...
  Saved: C:\Users\INFOTEC\OneDrive\Bureau\newton_week_to_day\India_Stock_Quotidien.xlsx  (466 produits)
Processing Korea...
  Saved: C:\Users\INFOTEC\OneDrive\Bureau\newton_week_to_day\Korea_Stock_Quotidien.xlsx  (107 produits)
Processing Kunshan...
  Saved: C:\Users\INFOTEC\OneDrive\Bureau\newton_week_to_day\Kunshan_Stock_Quotidien.xlsx  (241 produits)
Processing Tianjin...
  Saved: C:\Users\INFOTEC\OneDrive\Bureau\newton_week_to_day\Tianjin_Stock_Quotidien.xlsx  (211 produits)
Processing USA...
  Saved: C:\Users\INFOTEC\OneDrive\Bureau\newton_week_to_day\USA_Stock_Quotidien.xlsx  (469 produits)
Processing SAME...
  Saved: C:\Users\INFOTEC\OneDrive\Bureau\newton_week_to_day\SAME_Stock_Quotidien.xlsx  (44 produits)
P

In [ ]:
import os
import pandas as pd
import numpy as np
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ==============================
# CONFIG
# ==============================
W1 = r"C:\Users\INFOTEC\OneDrive\Bureau\anomalieW1\anomalie_metier\Week1_With_Anomalies_Metier.xlsx"
W2 = r"C:\Users\INFOTEC\OneDrive\Bureau\anomalie_w2\anomalie_metier\Week2_With_Anomalies_Metier.xlsx"
W3 = r"C:\Users\INFOTEC\OneDrive\Bureau\anomalie_w3\anomalie_metier\Week3_With_Anomalies_Metier.xlsx"
W4 = r"C:\Users\INFOTEC\OneDrive\Bureau\anomalie_w4\anomalie_metier\Week4_With_Anomalies_Metier.xlsx"

OUT_DIR = r"C:\Users\INFOTEC\OneDrive\Bureau\newton_week_to_day_1234"
os.makedirs(OUT_DIR, exist_ok=True)

PAYS_LIST = ['Cyclam', 'Germany', 'India', 'Korea', 'Kunshan', 'Tianjin', 'USA', 'SAME', 'SCEET']
DAYS      = ['Lundi', 'Mardi', 'Mercredi', 'Jeudi', 'Vendredi', 'Samedi']

# ==============================
# COULEURS
# ==============================
C_HDR_BG   = '1F4E79'
C_HDR_FG   = 'FFFFFF'
C_SUM_BG   = '1A5276'
C_ROW_ALT  = ['EBF5FB', 'FDFEFE']
C_MISSING  = 'E8E8E8'
C_PERT_OK  = 'C6EFCE'
C_PERT_KO  = 'FFC7CE'
C_DAY_HDRS = ['BDD7EE', 'FCE4D6', 'E2EFDA', 'FFF2CC', 'F4CCFF', 'D9EAD3']
WK_COLORS  = ['1A6BAD', '1A8A6B', '8A1A6B', 'B8860B']   # W4 = gold

ANOM_COLORS = {
    'Normal':               ('F2F2F2', '000000'),
    'Rupture de stock':     ('FFC7CE', 'C00000'),
    'Sur-stock critique':   ('FFEB9C', '9C6500'),
    'Rotation trop rapide': ('BDD7EE', '17375E'),
    'Sous-stock':           ('FCE4D6', '833C00'),
}

thin = Side(style='thin', color='BFBFBF')
BORD = Border(left=thin, right=thin, top=thin, bottom=thin)


def to_f(v):
    try:
        return float(str(v).replace(',', '.').replace(' ', '').replace('\xa0', ''))
    except:
        return 0.0


def sc(cell, bold=False, bg=None, fg='000000', align='center', wrap=False, fmt=None, size=9):
    cell.font      = Font(name='Arial', bold=bold, color=fg, size=size)
    if bg:
        cell.fill  = PatternFill('solid', start_color=bg)
    cell.alignment = Alignment(horizontal=align, vertical='center', wrap_text=wrap)
    cell.border    = BORD
    if fmt:
        cell.number_format = fmt


def newton_daily(inv_start, wu):
    xs    = np.linspace(1, 3, 6)
    d1    = 1.01 - 0.99
    d2    = 0.99 - 1.01
    d12   = (d2 - d1) / 2
    facs  = 0.99 + d1 * (xs - 1) + d12 * (xs - 1) * (xs - 2)
    raw   = (wu / 6) * facs
    if raw.sum() > 0:
        raw = raw * (wu / raw.sum())
    avant, apres = [], []
    s = inv_start
    for u in raw:
        avant.append(round(s, 4))
        s = max(0.0, s - u)
        apres.append(round(s, 4))
    usage = [round(a - b, 4) for a, b in zip(avant, apres)]
    return avant, usage, apres


def get_anom_cols(df_cols):
    mapping = {}
    for c in df_cols:
        cl = c.strip()
        if 'Inventory_Negatif' in cl or 'Inventory_Neg' in cl:
            mapping['inv_neg'] = cl
        elif 'Rupture_Stock' in cl or 'Rupture' in cl:
            mapping['rupture'] = cl
        elif 'Rotation_Rapide' in cl or 'Rotation' in cl:
            mapping['rotation'] = cl
        elif 'Sous_Stock' in cl:
            mapping['sous_stock'] = cl
        elif 'Surstock_Critique' in cl or 'Surstock' in cl:
            mapping['surstock'] = cl
        elif 'Total_Anomalies' in cl:
            mapping['total'] = cl
        elif cl == 'Anomalie_Metier':
            mapping['metier'] = cl
    return mapping


def load_pays_data(pays):
    """Load W1, W2, W3, W4 for a pays — return (df1, df2, df3, df4) with union of PNs"""
    raws = []
    for wpath in [W1, W2, W3, W4]:
        xl = pd.read_excel(wpath, sheet_name=pays, header=0)
        xl.columns = xl.columns.str.strip()
        xl['Part Number'] = xl['Part Number'].astype(str).str.strip()
        raws.append(xl)

    # Union ordered PNs
    seen = {}
    for df in raws:
        for _, row in df.iterrows():
            pn = str(row['Part Number']).strip()
            if pn not in seen:
                seen[pn] = {
                    'Description':    row.get('Description', ''),
                    'Supplier':       row.get('Supplier', ''),
                    'Unit Price (€)': to_f(row.get('Unit Price (€)', 0)),
                }
    all_pns = list(seen.keys())

    dfs = []
    for df in raws:
        rows = []
        for pn in all_pns:
            match = df[df['Part Number'] == pn]
            if not match.empty:
                r = match.iloc[0].to_dict()
                r['_missing'] = False
            else:
                r = {
                    'Part Number':          pn,
                    'Description':          seen[pn]['Description'],
                    'Supplier':             seen[pn]['Supplier'],
                    'Unit Price (€)':       seen[pn]['Unit Price (€)'],
                    'Real Inventory (Qty)': 0.0,
                    'Weekly Usage (Qty)':   0.0,
                    'Stock Value (€)':      0.0,
                    'Weekly Target':        0.0,
                    '_missing': True,
                }
            rows.append(r)
        dfs.append(pd.DataFrame(rows).reset_index(drop=True))
    return dfs, all_pns, seen


# ==============================
# WRITE WEEK SHEET
# ==============================
def write_week_sheet(wb, pays, week_num, df_wk, all_newton):
    ws = wb.create_sheet(f'Week{week_num}')
    ws.freeze_panes = 'E4'

    amap = get_anom_cols(df_wk.columns.tolist())
    detail_keys   = ['inv_neg', 'rupture', 'rotation', 'sous_stock', 'surstock', 'total']
    detail_labels = ['Inv. Neg', 'Rupture', 'Rotation rapide', 'Sous-stock', 'Sur-stock', 'Total Anom.']
    present_details = [(k, detail_labels[i]) for i, k in enumerate(detail_keys) if k in amap]

    N_ANOM_DETAIL = len(present_details)
    DAY_START = 10 + N_ANOM_DETAIL
    NCOLS = DAY_START - 1 + 6 * 5

    # Row 1: Title
    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=NCOLS)
    t = ws.cell(1, 1, f'{pays} — Week {week_num} | Stock Quotidien Newton | Anomalies Metier')
    sc(t, bold=True, bg=C_HDR_BG, fg=C_HDR_FG, align='left', size=10)
    ws.row_dimensions[1].height = 20

    # Row 2: Group headers
    for c, lbl in [(1,'Part Number'),(2,'Description'),(3,'Supplier'),(4,'Unit Price (€)')]:
        ws.merge_cells(start_row=2, start_column=c, end_row=3, end_column=c)
        sc(ws.cell(2, c, lbl), bold=True, bg=C_HDR_BG, fg=C_HDR_FG, wrap=True)
        sc(ws.cell(3, c), bg=C_HDR_BG)

    ws.merge_cells(start_row=2, start_column=5, end_row=2, end_column=8)
    sc(ws.cell(2, 5, f'Resume S{week_num}'), bold=True, bg='2E75B6', fg='FFFFFF')
    for c in [6,7,8]:
        sc(ws.cell(2, c), bg='2E75B6')

    ws.merge_cells(start_row=2, start_column=9, end_row=2, end_column=9 + N_ANOM_DETAIL)
    sc(ws.cell(2, 9, 'Anomalies Metier'), bold=True, bg='C00000', fg='FFFFFF')
    for c in range(10, 10 + N_ANOM_DETAIL):
        sc(ws.cell(2, c), bg='C00000')

    for d in range(6):
        sc_ = DAY_START + d * 5
        ws.merge_cells(start_row=2, start_column=sc_, end_row=2, end_column=sc_ + 4)
        day_cell = ws.cell(2, sc_, f'J{d+1} — {DAYS[d]}')
        day_cell.font      = Font(name='Arial', bold=True, color='1F3864', size=9)
        day_cell.fill      = PatternFill('solid', start_color=C_DAY_HDRS[d])
        day_cell.alignment = Alignment(horizontal='center', vertical='center')
        day_cell.border    = BORD
        for c in range(sc_+1, sc_+5):
            ws.cell(2, c).fill   = PatternFill('solid', start_color=C_DAY_HDRS[d])
            ws.cell(2, c).border = BORD
    ws.row_dimensions[2].height = 18

    # Row 3: Sub-headers
    for i, lbl in enumerate(['Inv. Reel','Usage Hebdo','Val. Stock','Cible']):
        sc(ws.cell(3, 5+i, lbl), bold=True, bg='1A5276', fg='FFFFFF', wrap=True)

    sc(ws.cell(3, 9, 'Type'), bold=True, bg='C00000', fg='FFFFFF', wrap=True)
    for i, (_, lbl) in enumerate(present_details):
        sc(ws.cell(3, 10+i, lbl), bold=True, bg='C00000', fg='FFFFFF', wrap=True, size=8)

    for d in range(6):
        for j, lbl in enumerate(['Avant','Usage','Apres','Valeur','Perturb%']):
            c = DAY_START + d*5 + j
            cell = ws.cell(3, c, lbl)
            cell.font      = Font(name='Arial', bold=True, color='1F3864', size=9)
            cell.fill      = PatternFill('solid', start_color=C_DAY_HDRS[d])
            cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
            cell.border    = BORD
    ws.row_dimensions[3].height = 28

    # Data rows
    for r_idx, row in df_wk.iterrows():
        pn         = str(row.get('Part Number', '')).strip()
        er         = r_idx + 4
        is_missing = row.get('_missing', False) == True
        bg         = C_MISSING if is_missing else C_ROW_ALT[r_idx % 2]
        up         = to_f(row.get('Unit Price (€)', 0))
        ri         = to_f(row.get('Real Inventory (Qty)', 0))
        wu         = to_f(row.get('Weekly Usage (Qty)', 0))
        sv         = to_f(row.get('Stock Value (€)', 0))
        wt         = to_f(row.get('Weekly Target', 0))

        def wc(col, val, fmt=None, bg_ov=None, fg_ov=None, bold=False):
            cell = ws.cell(er, col, val)
            cell.font      = Font(name='Arial', size=9, color=fg_ov or '000000', bold=bold)
            cell.fill      = PatternFill('solid', start_color=bg_ov or bg)
            cell.alignment = Alignment(
                horizontal='right' if isinstance(val, (int, float)) else 'left',
                vertical='center')
            cell.border = BORD
            if fmt:
                cell.number_format = fmt

        wc(1, pn)
        wc(2, str(row.get('Description', '')) if pd.notna(row.get('Description', '')) else '')
        wc(3, str(row.get('Supplier', ''))    if pd.notna(row.get('Supplier', ''))    else '')
        wc(4, up, '#,##0.0000')
        wc(5, ri, '#,##0.00')
        wc(6, wu, '#,##0.00')
        wc(7, sv, '#,##0.00')
        wc(8, wt, '#,##0.00')

        metier_val = ''
        if not is_missing and 'metier' in amap:
            v = row.get(amap['metier'], '')
            metier_val = str(v) if pd.notna(v) else ''
        anom_bg, anom_fg = ANOM_COLORS.get(metier_val, (bg, '000000'))
        if is_missing:
            anom_bg = C_MISSING
        wc(9, metier_val, bg_ov=anom_bg, fg_ov=anom_fg, bold=(metier_val not in ('', 'Normal')))

        for i, (key, _) in enumerate(present_details):
            col_name = amap[key]
            v = 0
            if not is_missing and col_name in row.index:
                v = to_f(row.get(col_name, 0))
            flag_bg = anom_bg if v == 1 else (C_MISSING if is_missing else bg)
            wc(10+i, int(v) if v in (0,1) else v, bg_ov=flag_bg)

        nt  = all_newton.get(pn)
        wu6 = wu / 6 if wu != 0 else 0
        for d in range(6):
            bc = DAY_START + d * 5
            if nt:
                av, us, ap = nt
                avant = av[d]
                usage = us[d]
                apres = ap[d]
                sv_d  = round(apres * up, 4)
            else:
                avant = usage = apres = sv_d = 0.0
            perturb = ((usage - wu6) / wu6 * 100) if wu6 != 0 else 0.0

            wc(bc,   avant, '#,##0.00')
            wc(bc+1, usage, '#,##0.00')
            wc(bc+2, apres, '#,##0.00')
            wc(bc+3, sv_d,  '#,##0.00')
            pc = ws.cell(er, bc+4, round(perturb, 2))
            pc.font      = Font(name='Arial', size=9)
            pc.fill      = PatternFill('solid', start_color=C_PERT_OK if abs(perturb) <= 2 else C_PERT_KO)
            pc.alignment = Alignment(horizontal='right', vertical='center')
            pc.border    = BORD
            pc.number_format = '0.00'

    ws.column_dimensions['A'].width = 16
    ws.column_dimensions['B'].width = 30
    ws.column_dimensions['C'].width = 22
    ws.column_dimensions['D'].width = 11
    ws.column_dimensions['E'].width = 11
    ws.column_dimensions['F'].width = 11
    ws.column_dimensions['G'].width = 11
    ws.column_dimensions['H'].width = 10
    ws.column_dimensions['I'].width = 20
    for i in range(N_ANOM_DETAIL):
        ws.column_dimensions[get_column_letter(10+i)].width = 11
    for c in range(DAY_START, NCOLS+1):
        ws.column_dimensions[get_column_letter(c)].width = 11


# ==============================
# WRITE SUMMARY SHEET  (4 semaines)
# ==============================
def write_summary(wb, pays, dfs, all_pns, newton_all):
    ws = wb.create_sheet('Summary', 0)
    ws.freeze_panes = 'E4'

    # Anomaly maps per week
    anom_maps = []
    for w, df in enumerate(dfs):
        df_c = df.copy()
        df_c.columns = [str(c).strip() for c in df_c.columns]
        amap = get_anom_cols(df_c.columns.tolist())
        m = {}
        if 'metier' in amap:
            for _, row in df_c.iterrows():
                pn = str(row.get('Part Number', '')).strip()
                v  = row.get(amap['metier'], '')
                m[pn] = str(v) if pd.notna(v) else ''
        anom_maps.append(m)

    N_WEEKS = 4
    # 4 info cols + 4*(5 data cols) = 4 + 20 = 24
    NCOLS = 4 + N_WEEKS * 5

    # Row 1: title
    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=NCOLS)
    t = ws.cell(1, 1, f'{pays} — Summary 4 Semaines | Stock + Anomalies Metier')
    sc(t, bold=True, bg=C_HDR_BG, fg=C_HDR_FG, align='left', size=10)
    ws.row_dimensions[1].height = 20

    for c, lbl in [(1,'Part Number'),(2,'Description'),(3,'Supplier'),(4,'Unit Price')]:
        ws.merge_cells(start_row=2, start_column=c, end_row=3, end_column=c)
        sc(ws.cell(2, c, lbl), bold=True, bg=C_HDR_BG, fg=C_HDR_FG, wrap=True)
        sc(ws.cell(3, c), bg=C_HDR_BG)

    for w in range(N_WEEKS):
        sc_ = 5 + w * 5
        ws.merge_cells(start_row=2, start_column=sc_, end_row=2, end_column=sc_+4)
        sc(ws.cell(2, sc_, f'Week {w+1}'), bold=True, bg=WK_COLORS[w], fg='FFFFFF')
        for c in range(sc_+1, sc_+5):
            sc(ws.cell(2, c), bg=WK_COLORS[w])
    ws.row_dimensions[2].height = 18

    for w in range(N_WEEKS):
        sc_ = 5 + w * 5
        for j, lbl in enumerate(['Inv. Reel','Usage','Val. Stock','Perturb% moy','Anomalie']):
            sc(ws.cell(3, sc_+j, lbl), bold=True, bg=WK_COLORS[w], fg='FFFFFF', wrap=True)
    ws.row_dimensions[3].height = 30

    for r_idx, pn in enumerate(all_pns):
        er  = r_idx + 4
        miss_all = all(
            dfs[w][dfs[w]['Part Number'].astype(str).str.strip() == pn].iloc[0].get('_missing', False) == True
            if not dfs[w][dfs[w]['Part Number'].astype(str).str.strip() == pn].empty else True
            for w in range(N_WEEKS)
        )
        bg = C_MISSING if miss_all else C_ROW_ALT[r_idx % 2]

        def wc(col, val, fmt=None, bg_ov=None, fg_ov=None, bold=False):
            cell = ws.cell(er, col, val)
            cell.font      = Font(name='Arial', size=9, color=fg_ov or '000000', bold=bold)
            cell.fill      = PatternFill('solid', start_color=bg_ov or bg)
            cell.alignment = Alignment(
                horizontal='right' if isinstance(val, (int, float)) else 'left',
                vertical='center')
            cell.border = BORD
            if fmt:
                cell.number_format = fmt

        row1 = dfs[0][dfs[0]['Part Number'].astype(str).str.strip() == pn]
        if not row1.empty:
            r = row1.iloc[0]
            wc(1, pn)
            wc(2, str(r.get('Description','')) if pd.notna(r.get('Description','')) else '')
            wc(3, str(r.get('Supplier',''))    if pd.notna(r.get('Supplier',''))    else '')
            wc(4, to_f(r.get('Unit Price (€)',0)), '#,##0.0000')

        for w in range(N_WEEKS):
            sc_ = 5 + w * 5
            row_w = dfs[w][dfs[w]['Part Number'].astype(str).str.strip() == pn]
            if not row_w.empty:
                rw  = row_w.iloc[0]
                ri  = to_f(rw.get('Real Inventory (Qty)', 0))
                wu  = to_f(rw.get('Weekly Usage (Qty)',   0))
                sv  = to_f(rw.get('Stock Value (€)',      0))
                wu6 = wu / 6 if wu != 0 else 0
                nt  = newton_all[w].get(pn)
                if nt and wu6 != 0:
                    perturbs = [((u - wu6) / wu6 * 100) for u in nt[1]]
                    pb = round(np.mean(perturbs), 2)
                else:
                    pb = 0.0
                wc(sc_,   ri, '#,##0.00')
                wc(sc_+1, wu, '#,##0.00')
                wc(sc_+2, sv, '#,##0.00')
                pc = ws.cell(er, sc_+3, pb)
                pc.font      = Font(name='Arial', size=9)
                pc.fill      = PatternFill('solid', start_color=C_PERT_OK if abs(pb) <= 2 else C_PERT_KO)
                pc.alignment = Alignment(horizontal='right', vertical='center')
                pc.border    = BORD
                pc.number_format = '0.00'
                anom_val = anom_maps[w].get(pn, '')
                ab, af = ANOM_COLORS.get(anom_val, (bg, '000000'))
                if rw.get('_missing', False):
                    ab = C_MISSING
                wc(sc_+4, anom_val, bg_ov=ab, fg_ov=af, bold=(anom_val not in ('','Normal')))
            else:
                for c in range(sc_, sc_+5):
                    wc(c, '-')

    # Total row
    tot_row = len(all_pns) + 4
    ws.merge_cells(start_row=tot_row, start_column=1, end_row=tot_row, end_column=4)
    sc(ws.cell(tot_row, 1, 'TOTAL / MOYENNE'), bold=True, bg=C_SUM_BG, fg='FFFFFF', align='center')
    for w in range(N_WEEKS):
        sc_ = 5 + w * 5
        for j in range(4):
            c     = sc_ + j
            col_l = get_column_letter(c)
            formula = (f'=AVERAGE({col_l}4:{col_l}{tot_row-1})' if j == 3
                       else f'=SUM({col_l}4:{col_l}{tot_row-1})')
            cell = ws.cell(tot_row, c, formula)
            cell.font      = Font(name='Arial', bold=True, color='FFFFFF', size=9)
            cell.fill      = PatternFill('solid', start_color=C_SUM_BG)
            cell.alignment = Alignment(horizontal='right', vertical='center')
            cell.border    = BORD
            cell.number_format = '0.00'
        wc_ = ws.cell(tot_row, sc_+4, '-')
        wc_.font      = Font(name='Arial', bold=True, color='FFFFFF', size=9)
        wc_.fill      = PatternFill('solid', start_color=C_SUM_BG)
        wc_.border    = BORD
        wc_.alignment = Alignment(horizontal='center', vertical='center')

    ws.column_dimensions['A'].width = 16
    ws.column_dimensions['B'].width = 30
    ws.column_dimensions['C'].width = 22
    ws.column_dimensions['D'].width = 11
    for c in range(5, NCOLS+1):
        ws.column_dimensions[get_column_letter(c)].width = 14


# ==============================
# MAIN
# ==============================
def main():
    for pays in PAYS_LIST:
        print(f'Processing {pays}...')
        try:
            dfs, all_pns, seen = load_pays_data(pays)
        except Exception as e:
            print(f'  SKIP {pays}: {e}')
            continue

        df1, df2, df3, df4 = dfs

        # Build newton per week (4 weeks)
        newton_all = [{}, {}, {}, {}]
        for pn in all_pns:
            for w, df in enumerate(dfs):
                row = df[df['Part Number'].astype(str).str.strip() == pn]
                if not row.empty:
                    ri = to_f(row.iloc[0].get('Real Inventory (Qty)', 0))
                    wu = to_f(row.iloc[0].get('Weekly Usage (Qty)',   0))
                else:
                    ri = wu = 0.0
                avant, usage, apres = newton_daily(ri, wu)
                newton_all[w][pn] = (avant, usage, apres)

        wb = Workbook()
        wb.remove(wb.active)

        write_summary(wb, pays, dfs, all_pns, newton_all)
        write_week_sheet(wb, pays, 1, df1, newton_all[0])
        write_week_sheet(wb, pays, 2, df2, newton_all[1])
        write_week_sheet(wb, pays, 3, df3, newton_all[2])
        write_week_sheet(wb, pays, 4, df4, newton_all[3])   # ← W4

        out_path = os.path.join(OUT_DIR, f'{pays}_Stock_Quotidien.xlsx')
        wb.save(out_path)
        print(f'  Saved: {out_path}  ({len(all_pns)} produits)')

    print('\nDone!')


main()